## Overview

This notebook integrates the completed Demographic, Behavioural Evidence, and Opportunity components into a final mobility representation. For each constrained representative population profile, it constructs trip-purpose probabilities and their corresponding temporal and travel-mode distributions. POI destination weights are then incorporated using the general Opportunity Layer and the industry-based refinement component. The integrated workflow is first tested on selected representative profiles before being applied to the full representative population space.

### 1. Import Libraries

In [19]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

### 2. Set Up Project Paths

In [20]:
project_root = Path.cwd().parents[1]

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

### 3. Import Main Inputs

In [21]:
representative_population_path = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_population_space.parquet"
)

behavioural_evidence_path = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)

opportunity_layer_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "opportunity_layer.json"
)

industry_opportunity_component_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "industry_to_poi_mapping.json"
)

In [22]:
representative_population = pd.read_parquet(
    representative_population_path
)

with open(
    behavioural_evidence_path,
    "r",
    encoding="utf-8",
) as file:
    assembled_behavioural_evidence = json.load(file)

with open(
    opportunity_layer_path,
    "r",
    encoding="utf-8",
) as file:
    opportunity_layer = json.load(file)

with open(
    industry_opportunity_component_path,
    "r",
    encoding="utf-8",
) as file:
    industry_component = json.load(file)

In [23]:
print(
    "Representative population shape:",
    representative_population.shape,
)

print(
    "Behavioural evidence purposes:",
    len(assembled_behavioural_evidence),
)

print(
    "Opportunity Layer entries:",
    len(opportunity_layer),
)

print(
    "Industry component entries:",
    len(industry_component),
)

Representative population shape: (2386560, 14)
Behavioural evidence purposes: 14
Opportunity Layer entries: 14
Industry component entries: 19


### 4. Import Main Functions

These functions integrate the Demographic Layer and Behavioural Evidence Layer and apply the relevant NTS constraints.

In [24]:
from src.demographic_behaviour_layers_integration import (
    construct_constrained_behaviour_profile,
    DEMOGRAPHIC_ATTRIBUTES
)

### 5. Bahavioural Evidence and Demographic Layers Integration with a Sample Population Profile

In [25]:
profile_index = 1500000 # For future use, this can be changed to select different profiles from the representative population

sample_representative_profile = (
    representative_population
    .loc[:, list(DEMOGRAPHIC_ATTRIBUTES)]
    .iloc[profile_index]
    .to_dict()
)

sample_behavioural_representation = (
    construct_constrained_behaviour_profile(
        representative_population=(
            sample_representative_profile
        ),
        assembled_nts_evidence=(
            assembled_behavioural_evidence
        ),
    )
)

print(
    sample_behavioural_representation.keys()
)

print(
    sample_behavioural_representation[
        "behavioural_representation"
    ].keys()
)

dict_keys(['demographic_profile', 'behavioural_representation'])
dict_keys(['trip_purpose_distribution', 'purpose_conditioned_distributions'])


In [26]:
integrated_representation = (
    construct_constrained_behaviour_profile(
        representative_population=sample_representative_profile,
        assembled_nts_evidence=(
            assembled_behavioural_evidence
        ),
    )
)

In [27]:
pd.DataFrame([sample_representative_profile])

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership
0,50-64,Male,Couple household: All children non-dependent,Economically active (excluding full-time stude...,L Real estate activities,Does not apply,5km to less than 10km,1 car or van in household


#### Check the trip distribution

In [28]:
sample_trip_purpose_distribution = (
    sample_behavioural_representation[
        "behavioural_representation"
    ]["trip_purpose_distribution"]
)

display(sample_trip_purpose_distribution)

{'Commuting': 0.1658049425,
 'Business': 0.0524143435,
 'Education': 0.00031256000000000003,
 'Escort Education': 0.037335577,
 'Shopping': 0.20824478070000002,
 'Other Escort': 0.072827616,
 'Personal Business': 0.08995064300000001,
 'Visiting Friends at Private Home': 0.0662559659,
 'Visiting Friends Elsewhere': 0.0531768137,
 'Entertainment or Public Activity': 0.0560380138,
 'Sport: Participate': 0.0165308992,
 'Holiday: Base': 0.015142675500000001,
 'Day Trip': 0.0425078539,
 'Just Walk': 0.12345731530000001}

In [29]:
sum(sample_trip_purpose_distribution.values())

1.0

In [30]:
sample_purpose_conditioned_distributions = (
    sample_behavioural_representation[
        "behavioural_representation"
    ]["purpose_conditioned_distributions"]
)

pd.DataFrame(sample_purpose_conditioned_distributions.keys(), columns=["Purpose"])

,Purpose
0,Commuting
1,Business
2,Education
3,Escort Education
4,Shopping
5,Other Escort
6,Personal Business
7,Visiting Friends at Private Home
8,Visiting Friends Elsewhere
9,Entertainment or Public Activity


### 6. Opportunity Layer Integration

In [31]:
from pprint import pprint

sample_purpose = "Commuting"

print("Behavioural representation:")
pprint(
    sample_purpose_conditioned_distributions[
        sample_purpose
    ]
)

print("\nOpportunity representation:")
pprint(
    opportunity_layer[
        sample_purpose
    ]
)

Behavioural representation:
{'temporal_distribution': {'0000 to 0059 (%)': 0.0023984035,
                           '0100 to 0159 (%)': 0.0014822167,
                           '0200 to 0259 (%)': 0.0011191196,
                           '0300 to 0359 (%)': 0.0021863867,
                           '0400 to 0459 (%)': 0.008278964,
                           '0500 to 0559 (%)': 0.0305873984,
                           '0600 to 0659 (%)': 0.0796545308,
                           '0700 to 0759 (%)': 0.15264470830000001,
                           '0800 to 0859 (%)': 0.1332867475,
                           '0900 to 0959 (%)': 0.03658019,
                           '1000 to 1059 (%)': 0.018253352100000002,
                           '1100 to 1159 (%)': 0.0163121804,
                           '1200 to 1259 (%)': 0.0219104564,
                           '1300 to 1359 (%)': 0.031365689700000005,
                           '1400 to 1459 (%)': 0.0348657682,
                           '1500 to 1

In [32]:
from copy import deepcopy


sample_final_representation = deepcopy(
    sample_behavioural_representation
)

sample_final_representation[
    "behavioural_representation"
]["purpose_conditioned_distributions"][
    sample_purpose
].update(
    opportunity_layer[sample_purpose]
)

In [33]:
pprint(
    sample_final_representation[
        "behavioural_representation"
    ]["purpose_conditioned_distributions"][
        sample_purpose
    ],
    width=60,
)

{'mapping_strategy': 'Representative Mapping',
 'poi_distribution': {'Employment': 1.0},
 'temporal_distribution': {'0000 to 0059 (%)': 0.0023984035,
                           '0100 to 0159 (%)': 0.0014822167,
                           '0200 to 0259 (%)': 0.0011191196,
                           '0300 to 0359 (%)': 0.0021863867,
                           '0400 to 0459 (%)': 0.008278964,
                           '0500 to 0559 (%)': 0.0305873984,
                           '0600 to 0659 (%)': 0.0796545308,
                           '0700 to 0759 (%)': 0.15264470830000001,
                           '0800 to 0859 (%)': 0.1332867475,
                           '0900 to 0959 (%)': 0.03658019,
                           '1000 to 1059 (%)': 0.018253352100000002,
                           '1100 to 1159 (%)': 0.0163121804,
                           '1200 to 1259 (%)': 0.0219104564,
                           '1300 to 1359 (%)': 0.031365689700000005,
                           '1400 to 1

In [34]:
missing_opportunity_purposes = [
    purpose
    for purpose in sample_purpose_conditioned_distributions
    if purpose not in opportunity_layer
]

print(
    "Missing Opportunity mappings:",
    missing_opportunity_purposes,
)

Missing Opportunity mappings: []


In [35]:
for purpose in (
    sample_final_representation[
        "behavioural_representation"
    ]["purpose_conditioned_distributions"]
):
    sample_final_representation[
        "behavioural_representation"
    ]["purpose_conditioned_distributions"][
        purpose
    ].update(
        deepcopy(opportunity_layer[purpose])
    )

In [36]:
final_purpose_distributions = (
    sample_final_representation[
        "behavioural_representation"
    ]["purpose_conditioned_distributions"]
)

pd.DataFrame([
    {
        "Shared Trip Purpose": purpose,
        "Mapping Strategy": details[
            "mapping_strategy"
        ],
        "POI Distribution": details[
            "poi_distribution"
        ],
    }
    for purpose, details
    in final_purpose_distributions.items()
])

,Shared Trip Purpose,Mapping Strategy,POI Distribution
0,Commuting,Representative Mapping,{'Employment': 1.0}
1,Business,Representative Mapping,{'Employment': 1.0}
2,Education,Direct Mapping,{'Education': 1.0}
3,Escort Education,Direct Mapping,{'Education': 1.0}
4,Shopping,Direct Mapping,{'Retail': 1.0}
5,Other Escort,Context-dependent Mapping,"{'Employment': 0.1478872700618409, 'Retail': 0..."
6,Personal Business,Context-dependent Mapping,"{'Healthcare': 0.149015490349917, 'Public Serv..."
7,Visiting Friends at Private Home,Excluded Mapping,{}
8,Visiting Friends Elsewhere,Representative Mapping,{'Food and Beverage': 1.0}
9,Entertainment or Public Activity,Representative Mapping,{'Recreation': 1.0}
